In [58]:
import torch
import torch.nn as nn

inputs = torch.rand([6,3])
print(inputs.shape)
print(inputs @ inputs.T)

torch.Size([6, 3])
tensor([[0.8471, 0.2288, 1.0007, 0.8010, 1.0214, 0.7282],
        [0.2288, 0.0894, 0.2456, 0.2339, 0.3325, 0.1864],
        [1.0007, 0.2456, 1.2058, 0.9624, 1.1566, 0.8844],
        [0.8010, 0.2339, 0.9624, 1.5532, 1.0247, 1.0466],
        [1.0214, 0.3325, 1.1566, 1.0247, 1.3485, 0.8674],
        [0.7282, 0.1864, 0.8844, 1.0466, 0.8674, 0.7993]])


# Simple per token attention mechanism

In [29]:
d_in = inputs.shape[1]
x_2 = inputs[1]
d_out = 2
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))
# with respect to the second token - that is why we have query
q_2 = x_2 @ W_query
k_2 = x_2 @ W_key
v_2 = x_2 @ W_value
print(q_2)
print(k_2)
print(v_2)

# only need KV for the rest of the token sequence
keys = inputs @ W_key
values = inputs @ W_value
print(keys)

attn_scores_2 = q_2 @ keys.T
print(attn_scores_2)

# normalize the scores to sum to 1 using softmax
d_k = keys.shape[-1] # = 2 (same as d_out)
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)



tensor([0.1303, 0.6850], grad_fn=<SqueezeBackward4>)
tensor([0.2337, 0.4687], grad_fn=<SqueezeBackward4>)
tensor([0.1020, 0.5687], grad_fn=<SqueezeBackward4>)
tensor([[0.3119, 0.9700],
        [0.2337, 0.4687],
        [0.4733, 1.2596],
        [0.2142, 0.4418],
        [0.4605, 1.0832],
        [0.1918, 0.2528]], grad_fn=<MmBackward0>)
tensor([0.7052, 0.3516, 0.9246, 0.3305, 0.8020, 0.1981],
       grad_fn=<SqueezeBackward4>)
tensor([0.1824, 0.1420, 0.2130, 0.1399, 0.1953, 0.1274],
       grad_fn=<SoftmaxBackward0>)


# Self Attention class

In [32]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim = -1)

        return attn_weights @ values
    
torch.manual_seed(789)
sa = SelfAttention(d_in=3, d_out=2)

print(sa(inputs))

tensor([[-0.0616,  0.1428],
        [-0.0621,  0.1423],
        [-0.0604,  0.1436],
        [-0.0620,  0.1418],
        [-0.0627,  0.1416],
        [-0.0624,  0.1416]], grad_fn=<MmBackward0>)


# Causal Self-attention

In [35]:
mask = torch.triu(torch.ones(6,6), diagonal=1)
print(mask)
tmp_attn = torch.rand([6,6])
masked = tmp_attn.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.1291,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.2814, 0.8556,   -inf,   -inf,   -inf,   -inf],
        [0.4517, 0.1223, 0.0137,   -inf,   -inf,   -inf],
        [0.4405, 0.1300, 0.6143, 0.5208,   -inf,   -inf],
        [0.8937, 0.5287, 0.4264, 0.4279, 0.8224,   -inf],
        [0.1193, 0.3741, 0.6131, 0.5162, 0.8599, 0.7378]])


In [54]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.register_buffer('mask',
                             torch.triu(torch.ones(context_length, context_length), diagonal=1))
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x) # b, num_tokens, d_out
        keys = self.W_key(x) # b, num_tokens, d_out
        values = self.W_value(x) # b, num_tokens, d_out
        attn_scores = queries @ keys.transpose(1,2) # b, num_tokens, num_tokens
        attn_scores.masked_fill(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) # b, num_tokens, num_tokens
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim = -1) # b, num_tokens, num_tokens

        return attn_weights @ values # b, num_tokens, d_out
    
torch.manual_seed(789)
csa = CausalSelfAttention(d_in=3, d_out=2, context_length=6)

# add a batch dim to inputs
b_inputs = inputs.view([1,inputs.shape[0], inputs.shape[1]])
print(b_inputs.shape)
print(csa(b_inputs))

torch.Size([1, 6, 3])
tensor([[[-0.0793,  0.0426],
         [-0.0783,  0.0440],
         [-0.0790,  0.0444],
         [-0.0783,  0.0446],
         [-0.0798,  0.0418],
         [-0.0798,  0.0436]]], grad_fn=<UnsafeViewBackward0>)


# Simplified Multi-head attention

By stacking self attention together

In [57]:
class MultiHeadAttentionStacked(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([CausalSelfAttention(d_in, d_out, context_length, qkv_bias)
                                    for _ in range(num_heads)])
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

mhas = MultiHeadAttentionStacked(d_in=3, d_out=2, context_length=6, num_heads=2)
mhas(b_inputs) # batch, num_tokens, d_out * num_heads = 4

tensor([[[ 0.0226, -0.3224, -0.6332,  0.2058],
         [ 0.0183, -0.3165, -0.6318,  0.2047],
         [ 0.0121, -0.3089, -0.6282,  0.2022],
         [ 0.0165, -0.3145, -0.6302,  0.2037],
         [ 0.0234, -0.3234, -0.6340,  0.2062],
         [ 0.0106, -0.3070, -0.6278,  0.2018]]], grad_fn=<CatBackward0>)

# Multi-head attention

Efficient way by splitting one weaight matrix into section by cleverly leveraging `.view()` and `.transpose()`
Much more efficient than the `for()` loop before as that is effectively sequential

In [84]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, qkv_bias=False):
        super().__init__()
        assert(d_out % num_heads == 0)
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = self.d_out // self.num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.register_buffer('mask',
                             torch.triu(torch.ones(context_length, context_length), diagonal=1))
        self.out_proj = nn.Linear(d_out, d_out) # another option projection layer
               
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x) # b, num_tokens, d_out
        keys = self.W_key(x) # b, num_tokens, d_out
        values = self.W_value(x) # b, num_tokens, d_out

        # now we do some clever slicing and dicing on the big matrix to split it into heads
        # d_out = num_heads * head_dim : so we are essentially splitting it out
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim) # b, num_tokens, n_head, head_dim
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # now we reshuffle it to make each head like a batch dimension
        # this computes attention scores independently on sequences that are [num_tokens x head_dim]
        queries = queries.transpose(1,2) # b, n_head, num_tokens, head_dim
        keys = keys.transpose(1,2)
        values = values.transpose(1,2)
        attn_scores = queries @ keys.transpose(2,3) # b, n_head, num_tokens, num_tokens
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill(mask_bool, -torch.inf)
        print(f"attn_scores {attn_scores.shape}")

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1) # b, n_head, num_tokens, num_tokens

        context_vec = attn_weights @ values # b, n_head, num_tokens, head_dim
        print(context_vec.shape)

        # now put the heads back together
        context_vec = context_vec.transpose(1,2) # b, num_tokens, n_head, head_dim
        print(context_vec.shape)

        # and stack all the heads to create a unified concatenated output embedding from all the heads
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        print(context_vec.shape)

        return self.out_proj(context_vec)

print(d_out)
mha = MultiHeadAttention(d_in=3, d_out=8, context_length=6, num_heads=2)
mha(b_inputs) # batch, num_tokens, d_out * num_heads = 4

2
attn_scores torch.Size([1, 2, 6, 6])
torch.Size([1, 2, 6, 4])
torch.Size([1, 6, 2, 4])
torch.Size([1, 6, 8])


tensor([[[-0.1424, -0.0507,  0.4170,  0.2514, -0.2119,  0.0262,  0.1571,
          -0.2241],
         [-0.1394, -0.0554,  0.4197,  0.2445, -0.2116,  0.0290,  0.1584,
          -0.2276],
         [-0.1431, -0.0482,  0.4150,  0.2558, -0.2118,  0.0236,  0.1560,
          -0.2223],
         [-0.1423, -0.0495,  0.4151,  0.2538, -0.2114,  0.0244,  0.1560,
          -0.2231],
         [-0.1413, -0.0534,  0.4196,  0.2470, -0.2123,  0.0285,  0.1585,
          -0.2261],
         [-0.1425, -0.0499,  0.4170,  0.2533, -0.2123,  0.0251,  0.1571,
          -0.2235]]], grad_fn=<ViewBackward0>)